# WEL01 · M03 기초 경보 규칙과 오경보·미탐 확인 (교수자·해설용)

- 본 노트북은 **미션3(M03) 전용 교수자·해설용** 노트북입니다 (`ROLE = 'solution'`).
- 원자료 생성·참고 구현·상황판 소스가 노트북에 내장되어 있어 별도의 `.md` 파일 없이 위에서부터 바로 실행할 수 있습니다.
- 각 STEP 코드 셀에는 모범 pandas 풀이, 경계 조건 예외 처리, 해설 주석이 포함되어 있습니다.

In [ ]:
from pathlib import Path
import base64,zlib,json,re,html,hashlib,io,time,ast,copy
import numpy as np,pandas as pd
from IPython.display import display,HTML

PROJECT_ID='WEL01'
ROLE='solution'
WORKDIR=Path('/content') if Path('/content').is_dir() else Path.cwd()
PROJECT_HOME=WORKDIR/PROJECT_ID
OUTDIR=PROJECT_HOME/'outputs'
SOURCE_BUNDLE=None
DASHBOARD_ASSETS=None
REFERENCE_CACHE={}
REFERENCE_EXECUTION_SERIAL=0

def _sha(value):
    return hashlib.sha256(json.dumps(value,ensure_ascii=False,sort_keys=True,separators=(',',':')).encode()).hexdigest()

EMBEDDED_BUNDLE_B64 = '...'
EMBEDDED_ASSETS_B64 = 'eNqNOl1z20aSfwVxNgERgwgpiaIEHuLzeu0938ZOKvLePsgqGQSGJCIQ4GJASjqGW/Za2XIib61u177IieSS7+zE3kvqFH8qVd6X/BwC+g/XPTMAAYrM5cEiMNPT3dPf3XD/lEXpKf3UO/26v1Gkzr87XlOv+4FNgiKsDOq+vdlvm0HT8fRSrW5aa83A73q2/iZpNGYb8zXLd/1Af7NcnanPWrWG74V6udLZeLesVSrS2cAxXVW+7Ie+tGR6VPrNR7JK4aFISeA0BvVuGPqe6nidbqiGZCM0A2L2GRLHawFIKED6VjegQKfjO15IgoHmmpt+N+zbDu3Ao94MHLuGf4ohacNKSIrAWLftUb08O9PZkNqO1zY3CiW13AiUGrwVW8RptoDZUqnXGmg0JJ2iZ/b62SuWq7P1uUVxxfWWE5Jax7RtFFF5AZAudjZqHZ86oeN7Og0da22zFvodEJTpOk0PLuk2YN0MwtpkahINA99rpteou761xoSIqiCMSkqyJCHNmVJnI4NAiCePYN2xwxbSequGQi0ybnSXNMJaokqpJJXLGeRlFFIZcNe48nXYlKjvOrb05lx93qxWxEYxMG2nS/Uqgo5kFQag1Q6ozwuz8jrB6bIJJlEEbSKkgZsrOZGTcn1h3kytqjRjzy4mlMUiB8kKsW267nQZ4jVdxyOpxrU5IYYiKquC8rQAHBjqo2Fw6ZVS0cwAhDSLogELErvlSonpgWySeuCv9wVrc4vzi9WFLPEZJE5CsNkiiMdCfFppgbRTklKr3B/BI60xZmcridJA5Ki2UoZhqTWTPc2ZTDW8kIOczUCW5ydAUsvvkOQq89U5u7yY8/hGo1Gx6yOTKaHJzKUmk1hGBjVGAqk00NAzf46vjrxUq4KfJl47Mw+kuOc2zQ73iowGuUt0TI+4OQVmeOfOe9K07SopkXHTXsz6xeKIVt0HC24z8gMtNOsuKVIr8MH2/B4JGq6/rpvd0GdmItQ3V0HeGGx/ZMWu2aFETx6y7pq3nEHYUkO7P4Fti8yRhRPOnTBdFbGJ3ZrZHdE9fz0wO4Ax724LjVJjdnIQG4QY/KUw0L2wVbRajmsXSI94Sg5Do9owG3WQh++7dTNItdxwyQbTFiqLh0Ngpk11i2AEr+F+EVnS8U9qMDPcYAS2JLyNWeEERdbnrarVGFNkJaPICpfJCDVLOv3s9sLE6CcwCyXNsXDRJmHgWD8r+zQCiRlFIoo0t3JbGtEhM2SRWDkdlrL2MDsiLNVTi+MiNL3NdUiXJGsRARognOiGxO6PRSTh45V6tVy1wHVcP+xnrFAYL+M7H1YzXldmknD9Zj9rZp2AMJZqUxgci8sZV5lFV6nlPGmgeX5IspxlEzfLVgFhyOAUWK7p/lzLmMtYBo99oRl2aRL8ZiqV6lyVM7vOydV91x5oDR8kNXaHJGAuVK2F6kkJUbNH/r+qQnBd+gn7xdzPYsI/t4ntmIVRLlpAwSn9NIdlSxQReqfaZlpITYFgMT5XO2VKpZQQOs/8pLIklyqquWyUy3ogxsHgqnfV0xwPSqKuhcGIZgV9Mi8uVKbY2CCHRGrNJfXrzBxLoEwsORAen8d9LQdi+Tbp5+NpAGXHiSpkrNCYzmMqxRRgU9j8BNZ4suGq0hvOBrHHwVrq2II9gdtpzFz1TqmnPsY2oFBQjPf6cpcSLE0dK5TBTgGvZBnrjmf769rvzr9fKq+e++DyhYu/xspGMoFijxgltWM28afhuBDfDTk5iS5MjbNBYG4WKooG225BlhWxGzpgXU1qLK+IBeIahdBsqpZLWUeA/PCdDcP2rW4brEezoE0IyXmX4BuCKzWnUYAjyoZmuSall802MeAdlxHLG4YBvkcaoB0bYHDpHLdEYwnu6TUZlALhJOwGnrQxENyYnc6IapOEguQvNy/aBRk203tw3RgE7mY7PVmV+UK6D5pmm/ALm4nyYRv+apSEZ0NgA7yGAFqskF2zTlyAjP73KP50V4q2Hw+fbQlwoEs8yMWAjvcPsup1XVe1tLZDKahfSai2TcdjZPEBsAnvAzycvQQVYFURRKnBQrLIQRJU3KOZoq56ja7H7EwCi/Bc30QMbaL2TLdLFKEuc5q6ZBPom1orIA3jtx+9L3Y/qH9MrBDeCx5Zl37p+vXC8r8ufXBZo0w/TmOzwNDzq84oK2o/3OwQHbXgQuhHdt79mPqePFAQfcKZgZzBuwVAawWlBrK+4rQJ3IzZOjIQkJ6/lmGA86aokHFKyiBzW9ry16+gKxZ4p6OKOKlCC0CTe6NrZewgWyeOrB4XGRB7StdbBBhmy/gkK2oY8NeAgTBikIOC86bVKqwZ74VB1hjCljCENQVEgBiS3RAqZ0YpWcDNVLNQ5XEq+ASEgECBbwWS35Cyl4PnEUMjuDWEE/wloD0jWF5bqcGJHI+24LFnGAY+fPIJPqXOeQYt/vje3Xj/SNZRwYAYAWSfqUc+M24Tit5TFDamSMgARWWQuy3uKjVUTCoQ3FZqXI3JIgLk9M2rrUTZa2pvXAxcxRwMBDJ2Vwi9XqoRFVfqyeWVcdqM6Qxp2INCoJAQpIaFJUqT0GUeb1dqwiFTcyjUVQeDJUQ8BwTGwZT6pOAi2m8RiIBx4kLAr4MjtCFDTINWBkoNgwSAQZVgkXPYEcBOQSxnbs7lIppjWb32i76ldQIfNXjRHkg/vpJwRUSr9L3ldwM6iLf3hodbuBZvP4wP9qQPN8MWSATejv/zM2n49HD47PU1Ls9WOQ19oROCRtlqJ12k0GzAHXxlGous5YXf4YudeO9B/OUT6fjuk3jreym+vxP9923GxTdbEH2l+C787LKFw6PhswMp+vvj6M9Hgi1gCU7eiz9/AQfj/V0JVG+bVHAbPXotDV98G73cAuzR9q1o+6GGC/GDQ4C4Hn+9H+9fl+IXe/F/fSp9eHZpSTq+vRMfgBM853TviUNyGtmxXsuYIL5CuMAuMF1VVNYCjN7FUdYos1VKmLHBebYmo1HCr2aboQlmo3EbWgIbSbIkXzldTiAzAm3NCLFfW7py/kNQ6PgR6NDtJRyGFWZUuQTmJAEQ5YobCIUKh8KehZkNBSPx4I0OQHhSdHgIUmE6uLnPtPTFc4YDojk0c+dYAGLmFH+1g8aCe2CB56DwDgfDwz0pevY8fvD8msLqBaq5xAw8EvxL2HYTT8tUUWMyytZXIKrMK5ReiOXKpfeNHM68kDIHlAEnxmslYhuMTarx90JgvPcGf/zkEyHFYFnmd7r4K3lFAYKW27UJLXCw1CywuaXZ7MObXbAFaHfZOmt7GfsdjblxCxobrNcS9Mwib34mM4ifqEzG4TlKlqMNzhVb8D2rZXpQGbLKUhSHKWSNF421JNwldRe0kbxe4lFOWJYMDhEfbElAF27kQcU2AQb8JN6/zWFqiEeDFhbDvW0wYgaQw6Oj5QKug4G+M1t6z0hUAnr0mmGLY4BLYPnA74DQxWKGZYbtBMjp0xkQppeRIXRURMtuoI4yRcbuEQGwc7o8+MMv+pfMsIV+UMgwqo7xCd70Lpj72OoApHBNGXNWxgsUQpliBnZVqiXVTIqEwo1IQfCiZqgrE/y/w9IJdtGpcfz4Cn6jZ0fDw7/++Cq6tRUf3BARkXluLgJK8b0tiLgY4TCKJZgZmcS6KSQIi0wJXXwzF5FmU7v5akcE9Oj2daQpxU//R5pnNpKVBMeBiYM9nBMVF5dDSV1U0q2P0F2T9fkxpjmIkmmApvCMW5M5Hj6/Hh9ySe1DutmShkeH0YP9UQJIvpbwcky8AGpECZiTlTEf3zo4vrl/fHdPGr56Pfz+UGXp6/Zj0E5SdUnR0zvx3pYqJX7EVpGRaPt5tP1DvJ/wcnx3N946ir+8o8kjctz/Wc+XVirppu+x6MN8JAdi5M/n5JJspUZg9kjO7WVcwRzOuJKwQJTig+vx/UdoFbCX8820Y7nGO9lsIbIaEMsPbA3biGtqP61Y9Ez1ogpgPT2mcnVfgDBNgg5E6xD2GqM3lV1V7TqXiEm7AWuCqC4aX7VNTA9HJHJaPkhc90LKEPKiJ4dgBbtovuA9kAF/fAX1CkCJMkHUJtD45ASHV8/bJe6mYuy222awOc2Z+O4U20yLpfjmjeN7T4ALtEtRLYujqny8uxXf/zZpXTGqJaUAxqrKtZMnRm6Knpn3NRHmTpwZxRNZzRcCE/Bz5Lx4kFk6iZ7eAhPfHT79R/zFTry1l1R38T/uRH/bE9WdPAGVAONFafz8lqxe7rbrEO0tHNKaUCQ3IIZ6FlkF6fqeTRUt9C/g6KYwq4ymBpBLqdE3Pb9tupurFhYqOrjpa8YKZLubN4DG3vEXu1J8axdEAg08dJIpINSIACGCWhawGZg2WUUqDtT4nrWpy9Hfvh8eAa/7r9HAUAABcU3UyGrDDMD2bV6c6PKFsx9Jw8Od+OXd6NEBgDl0DV3DdF0gubfFiNz/FvW791hW2Twq3QerjR9CMXsCrNuBspKsor2vtqkui7J+ePh9vH0gHf/ls+GrA6lNR4DmxgQ4KJLhtgyOZ1cGI7QxfHUUfQ2B6uB29Oc70Tc3GBgbXAODicgOH4NFRjsg3i/vxE+/4+Iyu7YTrkLwcezVOoGWioC8nnGpQsERPdgD3CJTyWpotjtA2SYhOA6xU1Dcf7mX7qdfVVJUe1vxD0CtUV5tQjuy2umArMtwtT0IzNJbkEEtaBscdn3B7vHNHSiupOOvtqB0TWygRwJIxXCXb66j+x/ffQwbiS1GP3B5Wy1iraW3hq7kKZw/2AEe4cB30d+fcGwdk1K8whjEn14wr0IIMI7U0iBlY3MGBiKnpRoJ8JOOgWObJRIWluUTtgeuNtnUcGNkXPCWtSU2r+JXxR2oYleyM4flNZW5zwrOHvjwRgM+AgeqYu5XymgUwZ0S5xFQ9+cGDnz+gIFNQeeCboMkERLKAs8Q19NaJi2sKWcKwst7yjvlUmnk02XltPxWfmLhMUj57bff4Gc0h170QtJkp8/00qMzSu5YHcoz8BH5TKF3Ro53b8kgfIzvt+Ldh7Ki98ZDEburyphVBhPidmdUWIrQhO3dKut8IRCxHjFNGfx4miMQZlqGwL0p+YEXdLyQiO9dl6I/PsYGV4x10iZ9jK22bxN31eyCuwe05XSU3ISJbKKel2X8/x9gYMjNqgs5FKcYzG/5UtLQi7SKLRBLtKtdj4EmFpUCcs9nVdOKgkNkYSlkc0U5cccTTHO4CdLIQMq/vSiSIMYSSDYSH3Gwnp+POI5vPI++/g69XIqPHoCvQW2FWT9T0kB6ElVNZoaQS1Y4dohf3okeHEB9IKIlI8SLt4fRI4hgh3+N73+amSbk1c9sopZKQWMxhL79du41ScX9pBClIZ1iJGxvSkmeJFCesyDzb++dKHKhS2KY4RdH6j7oc2z+l+MsmQ2PEYbTYxdl28oApybptBvqJD4wyc+KEEaIBD+Tjt+erU0bpjD4n749huyk8mBlCS/OMY/JOQeggcWGrJyLdFrRbvKuvo2mjlvYh7ebGoAb8I89m25oJNVYnjBYTDLv2j443oXif3cn+vyOBCvR548SPvIXAYzKICshzpCYaICOpkgDdibLIs8ZZBhYSJMZS7xJyBhZgVA7DW38RJHXGACkEQa/10LpeelXEuTK6NsjNmfM1OaDzJwRXCt69h/QCr28jiUGeCFLuV/8CdogbKJzH8D+7eL5360uXTl75bzRP9kRjAZnelLzqskcSh+NpNRsuarnq1coqwIaYumri+FQ0m3OKlN6iUENJ8hgMfhZzjFKNeefKjXn9OnEWOrZtkmp1dM5H81M+Bwc7tWnjLHHRnvOpKmemOSxz6GjQSyfXTsrYk6LJHITExFLMN028FslFM6a568XlJr4vugko6LR98VkvAIPv++CR5/1nDbLAhcCs03Y553pO30hNK3Tpa1CH0Wgw32Ad/w/OfYlqp9gpRiqLELqMmO8GPrFcN0vmgnyYgOxU+zDptiKlleckShOUQbpPJ/xU8996asrg/SuAwX+XPVODf4PT6yS7w=='

def _decode_b64(payload):
    return json.loads(zlib.decompress(base64.b64decode(payload, validate=True)))

def load_embedded_bundle():
    global SOURCE_BUNDLE, DASHBOARD_ASSETS, REFERENCE_CACHE, REFERENCE_EXECUTION_SERIAL
    bundle = _decode_b64(EMBEDDED_BUNDLE_B64)
    loaded_assets = _decode_b64(EMBEDDED_ASSETS_B64)
    if bundle.get('projectId') != PROJECT_ID or bundle.get('version') != 'wel01-integrated-2':
        raise ValueError('프로젝트·버전 불일치')
    if bundle.get('fingerprint') != _sha({k: v for k, v in bundle.items() if k != 'fingerprint'}):
        raise ValueError('원자료 지문 불일치')
    if bundle.get('assetFingerprint') != _sha(loaded_assets):
        raise ValueError('화면 소스 지문 불일치')
    for m_id, item in bundle['missions'].items():
        if len(item['stages']) != 5:
            raise ValueError(m_id + ' 5단계 누락')
        if sum(s['minutes'] for s in item['stages']) != item['minutes']:
            raise ValueError(m_id + ' 시간 불일치')
        if hashlib.sha256(item['referenceCode'].encode()).hexdigest() != item['referenceCodeSha256']:
            raise ValueError(m_id + ' 참고 코드 지문 불일치')
    namespace = {}
    exec(compile(bundle['runtimeSource'], 'WEL01_reference_runtime.py', 'exec'), namespace)
    bundle['_reference_runner'] = namespace['_reference_table']
    SOURCE_BUNDLE, DASHBOARD_ASSETS, REFERENCE_CACHE = bundle, loaded_assets, {}
    REFERENCE_EXECUTION_SERIAL = 0
    print(f'WEL01 M03 내장 교육 자료 연결 완료 (5단계 · {bundle["missions"]["M03"]["minutes"]}분)')
    return bundle

def _connected():
    if SOURCE_BUNDLE is None:raise RuntimeError('먼저 첫 실행 셀을 실행해 내장 교육 자료를 로드하세요.')

def _parameter_code(mid,code,parameters):
    allowed=SOURCE_BUNDLE['missions'][mid]['parameters']
    if set(parameters)-set(allowed):raise ValueError('정의되지 않은 변경 파라미터')
    if mid=='M02':
        value=parameters['FLATLINE_RUN']
        if not isinstance(value,int) or value<1:raise ValueError('고착 분은 양의 정수')
        code=code.replace('def flatline(series, run=8):',f'def flatline(series, run={value}):')
    elif mid=='M05':
        value=float(parameters['SCORE_THRESHOLD'])
        if not 0<=value<=1:raise ValueError('문턱은 0~1')
        code=code.replace('eng = (test["score"] >= 0.5).values',f'eng = (test["score"] >= {value!r}).values')
    elif mid=='M09':
        value=float(parameters['MIN_RECALL'])
        if not 0<=value<=1:raise ValueError('재현율 하한은 0~1')
        code=code.replace('(g["재현율"] >= 0.90)',f'(g["재현율"] >= {value!r})')
    inserts={}
    for node in ast.parse(code).body:
        if isinstance(node,ast.Assign):
            names=[]
            for target in node.targets:
                names.extend(x.id for x in ast.walk(target) if isinstance(x,ast.Name))
            updates=[f'{key} = {parameters[key]!r}' for key in names if key in parameters and key not in ['MIN_RECALL','SCORE_THRESHOLD','FLATLINE_RUN']]
            if updates:inserts[node.end_lineno]='\n'.join(updates)
    lines=code.splitlines()
    for pos,value in sorted(inserts.items(),reverse=True):lines.insert(pos,value)
    return '\n'.join(lines)+'\n'

def source_table(mid,parameters=None,refresh=False):
    global REFERENCE_EXECUTION_SERIAL
    _connected();item=SOURCE_BUNDLE['missions'][mid]
    p=dict(item['parameters']);p.update(parameters or {})
    key=(mid,_sha(p))
    if refresh or key not in REFERENCE_CACHE:
        tuned=None
        if mid=='M10':
            # 현재 M9 실행 결과를 우선 사용  아직 실행하지 않았으면 기본 M9를 실행한다.
            matches=[(k,v) for k,v in REFERENCE_CACHE.items() if k[0]=='M09']
            if not matches:
                source_table('M09');matches=[(k,v) for k,v in REFERENCE_CACHE.items() if k[0]=='M09']
            m9=max(matches,key=lambda x:x[1][1]['executionOrder'])[1][1]['selected']
            tuned={'heat_temp':float(m9['th_heat']),'cold_temp':float(m9['th_cold']),
                'min_persist':int(m9['persist']),'wind_corr':float(m9['corr'])}
        code=_parameter_code(mid,item['referenceCode'],p)
        raw,meta=SOURCE_BUNDLE['_reference_runner'](mid,code,tuned)
        REFERENCE_EXECUTION_SERIAL+=1
        meta['executionOrder']=REFERENCE_EXECUTION_SERIAL
        meta.update(parameters=p,md_fingerprint=SOURCE_BUNDLE['fingerprint'],reference_name=mid+' 교육용 참고 구현')
        REFERENCE_CACHE[key]=(raw,meta)
        if mid=='M09':
            for stale in [k for k in REFERENCE_CACHE if k[0]=='M10']:del REFERENCE_CACHE[stale]
    raw,meta=REFERENCE_CACHE[key]
    out=raw.copy(deep=True);out.attrs['reference_meta']=copy.deepcopy(meta)
    return out

def TODO(mission,step,instruction):
    raise NotImplementedError(f'{mission} STEP {step}: TODO를 실제 pandas 코드로 완성하세요. {instruction}')

def _rows(df):
    return json.loads(df.to_json(orient='records',force_ascii=False,date_format='iso'))

def finish_mission(mid,raw,stages,stage_specs):
    _connected();canonical=SOURCE_BUNDLE['missions'][mid]['stages']
    if stage_specs!=canonical:raise ValueError(mid+' 내장 기준과 노트북 STEP 정의 불일치')
    if len(stages)!=5 or not isinstance(raw,pd.DataFrame) or raw.empty:raise ValueError('5단계와 원자료 필요')
    expected=list(raw.columns);original_ids=raw['원행ID'].tolist();report=[];cumulative=[]
    for k,(df,spec) in enumerate(zip(stages,stage_specs),1):
        if not isinstance(df,pd.DataFrame):raise TypeError(f'{mid} STEP{k}: DataFrame 필요')
        col=spec['outputColumn'];cumulative.append(col)
        if df['원행ID'].tolist()!=original_ids or not df.index.equals(raw.index):
            raise ValueError(f'{mid} STEP{k}: 행 누락·재정렬·중복이 발생했습니다.')
        if any(x not in df.columns for x in expected+cumulative):raise ValueError(f'{mid} STEP{k}: 필수 열 누락')
        if not df[expected].equals(raw[expected]):raise ValueError(f'{mid} STEP{k}: 원자료를 변경했습니다.')
        if df.columns.duplicated().any():raise ValueError('열 이름 중복')
        preview=df[['원행ID']+cumulative]
        report.append(dict(step=k,title=spec['title'],goal=spec['goal'],outputColumn=col,
            learnerHtml=spec.get('learnerHtml',''),rowCount=len(df),columns=list(preview.columns),rows=_rows(preview),minutes=spec['minutes']))
    location=OUTDIR/mid;location.mkdir(parents=True,exist_ok=True);stem=f'WEL01_{mid}'
    stages[-1].to_csv(location/(stem+'_analysis.csv'),index=False,encoding='utf-8-sig')
    meta=copy.deepcopy(raw.attrs['reference_meta']);plots=meta.pop('plots',[])
    details=dict(projectId='WEL01',mission=mid,originalRows=len(raw),codeSteps=report,
        referenceMeta=meta,mdFingerprint=SOURCE_BUNDLE['fingerprint'],
        structuralValidation='원행과 원본값·누적 열 검증 완료  코드의 의미·교관 PASS 판정과 별개')
    (location/(stem+'_code_report.json')).write_text(json.dumps(details,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
    config=dict(projectId='WEL01',mission=mid,title=SOURCE_BUNDLE['missions'][mid]['title'],
        scenario=SOURCE_BUNDLE['missions'][mid]['scenario'],hours=SOURCE_BUNDLE['missions'][mid]['minutes']/60,
        stages=report,sourceColumns=list(raw.columns),sourceRows=_rows(raw),meta=meta,plots=plots,
        fingerprint=SOURCE_BUNDLE['fingerprint'],role=ROLE)
    safe=json.dumps(config,ensure_ascii=False,default=str).replace('<','\\u003c')
    document='<!doctype html><html lang="ko"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>'+stem+'</title><style>'+DASHBOARD_ASSETS['css']+'</style></head><body><div id="app"></div><script>window.WEL01_CONFIG='+safe+';</script><script>'+DASHBOARD_ASSETS['js']+'</script></body></html>'
    path=location/(stem+'_result.html');path.write_text(document,encoding='utf-8')
    if not globals().get('HEADLESS_VALIDATION',False):
        display(HTML('<iframe title="'+stem+'" sandbox="allow-scripts allow-downloads" style="width:100%;height:1150px;border:0" srcdoc="'+html.escape(document,quote=True)+'"></iframe>'))
    print('HTML:',path);print('행 보존·누적 열 검사 완료  최종 PASS는 교관 판정')
    return path

load_embedded_bundle()

## M03 기초 경보 규칙과 오경보·미탐 확인

90명의 합성 자료를 5분 구간으로 나눈 4950행에 기초 경보 규칙을 적용합니다. 열 경보에는 체온과 심박 두 조건이 필요하고 한랭 경보는 체온 조건을 사용합니다. 유형을 맞힌 정도와 위험을 찾아낸 정도를 구분해 계산하고, 오경보와 미탐이 발생한 행을 원래 수치에서 확인하세요.

실습 시간 150분. 준비 셀 → STEP별 설명과 코드 → 마무리 셀 순서로 실행합니다.


In [ ]:
MID = 'M03'
PARAMETERS = {'TH_HEAT': 38.0, 'TH_COLD': 36.3, 'HR_GATE': 95}
raw = source_table(MID, parameters=PARAMETERS, refresh=True)
print("입력 표의 행·열 수:", raw.shape)
display(raw.head(6))


### M03 STEP 01 — 체온과 심박으로 기초 경보 유형 구하기

<h4>이 단계에서 해결할 일</h4><p>이동 중 체온이 높아졌다는 이유만으로 모든 구간에 같은 경보를 붙이면 심박 조건을 놓칠 수 있습니다. 한 행에 있는 5분 체온과 심박을 함께 읽고, 교육용 고정 규칙에 따라 열·한랭·경보 없음 중 하나를 표시합니다.</p><h4>시작 전 준비</h4><p>각 미션의 STEP 1~5는 앞 단계 결과를 이어서 사용합니다. WEL01_m03_student.ipynb의 첫 실행(환경 준비) 셀과 이 미션의 준비 셀을 순서대로 실행하면 현재 참고 프로그램의 결과가 raw라는 입력 표로 만들어집니다. PARAMETERS는 기본값을 유지합니다. 이 단계만 다시 실행하려면 그 미션의 준비 셀부터 바로 앞 STEP까지 순서대로 실행해 둡니다.</p><p>이번 입력 표는 <code>raw</code>, 완성한 결과 표는 <code>step1</code>입니다. 새로 만들 열은 <code>기초_경보유형</code>이며 배정 시간은 30분입니다. 노트북의 <code>M03 STEP 01</code> 코드 셀에서 TODO 호출을 본인이 작성한 pandas 계산으로 바꾸세요. 결과는 DataFrame이어야 합니다.</p><h4>입력값의 뜻</h4><ul><li><code>temp5</code> — 5분 구간의 체온 특징값(℃)입니다.</li><li><code>hr5</code> — 같은 구간의 심박 특징값(회/분)입니다.</li><li><code>PARAMETERS</code> — 기본값은 TH_HEAT=38.0, HR_GATE=95, TH_COLD=36.3입니다.</li></ul><h4>수행 순서</h4><ol><li>raw의 같은 행에서 temp5와 hr5를 읽습니다. 앞사람 체온과 뒷사람 심박을 섞지 않습니다.</li><li>체온이 TH_HEAT 이상이면서 심박이 HR_GATE 이상이면 heat로 표시합니다.</li><li>위 조건에 해당하지 않고 체온이 TH_COLD 이하이면 cold, 그 밖에는 normal로 표시합니다.</li><li>새 열 기초_경보유형을 만든 표를 step1에 저장하고 각 유형의 개수를 확인합니다. 기본값 평가를 마친 뒤 임계값 변경 실험은 별도로 기록합니다.</li></ol><h4>결과를 읽는 방법</h4><p>heat와 cold는 예시 프로그램의 경보 분류이며 normal은 이 규칙에 따른 경보가 없다는 뜻입니다.</p><h4>직접 확인할 가상 예시</h4><p>다음은 규칙을 이해하기 위한 가상 사례입니다. 실제 실행값을 대신 제출하는 자료가 아닙니다.</p><table><thead><tr><th>가정한 입력</th><th>기대 결과</th><th>이유</th></tr></thead><tbody><tr><td><code>{"temp5": 38.0, "hr5": 95}</code></td><td><code>heat</code></td><td>두 경계값을 모두 포함합니다.</td></tr><tr><td><code>{"temp5": 38.1, "hr5": 94}</code></td><td><code>normal</code></td><td>열 경보의 심박 조건을 만족하지 않습니다.</td></tr><tr><td><code>{"temp5": 36.3, "hr5": 80}</code></td><td><code>cold</code></td><td>한랭 경계값을 포함합니다.</td></tr></tbody></table><h4>완료 전 확인과 기록</h4><p>원행ID는 계산 전후 같은 자료를 찾는 번호입니다. 입력의 행 수·행 순서·인덱스·원래 열의 값을 유지하고, 앞 단계에서 만든 열도 모두 남깁니다. 각 행의 계산 결과는 새 열에 넣습니다. 실제 불리언 True·False와 문자열 "True"·"False"는 다릅니다. 값이 없다는 뜻의 결측값을 False나 0으로 대신 채우지 않습니다.</p><p>계산 뒤 해당 출력 열과 입력 열을 원행ID로 함께 확인합니다. 실제 자료에서 해당 사례가 있으면 최소 한 행의 입력·출력·판단 이유를 기록하고, 해당 사례가 없으면 0건 집계와 아래 가상 예시 시험 결과를 남깁니다. 관찰 내용은 5단계 완료 후 HTML의 해당 STEP 기록란에 옮기고 기록 JSON 저장을 눌러 보관합니다. HTML 입력란의 글은 페이지를 닫기 전에 저장해야 합니다.</p><h4>주의할 점</h4><p>체온이나 심박이 없으면 normal로 채우지 않습니다. 누락된 원행ID를 기록하고 입력을 확인받습니다. 임계값은 진단 기준으로 사용하지 않습니다.</p>


> **교수자·해설용 FAIL 점검 포인트**: 열 조건을 OR로 연결하거나, 경계값을 제외하거나, 체온만으로 heat를 부여하면 FAIL입니다.

In [ ]:
# M03 STEP 01 — 체온과 심박으로 기초 경보 유형 구하기 (교수자·해설용 예시 풀이)
# 해설: temp5 >= TH_HEAT AND hr5 >= HR_GATE -> heat, 그 외 temp5 <= TH_COLD -> cold, 나머지 -> normal.
# 결측값이 있는 행은 normal로 채우지 않고 결측을 유지합니다.
step1 = raw.copy()
th_heat = float(PARAMETERS.get("TH_HEAT", 38.0))
hr_gate = float(PARAMETERS.get("HR_GATE", 95))
th_cold = float(PARAMETERS.get("TH_COLD", 36.3))

valid = step1["temp5"].notna() & step1["hr5"].notna()
cond_heat = valid & (step1["temp5"] >= th_heat) & (step1["hr5"] >= hr_gate)
cond_cold = valid & (~cond_heat) & (step1["temp5"] <= th_cold)
step1["기초_경보유형"] = np.where(
    ~valid, None, np.where(cond_heat, "heat", np.where(cond_cold, "cold", "normal"))
)
display(step1[["원행ID", "기초_경보유형"]].head(10))

### M03 STEP 02 — 계산한 경보 유형이 합성 자료의 기준 유형과 같은지 확인하기

<h4>이 단계에서 해결할 일</h4><p>경보가 울렸는지와 열·한랭 유형을 맞혔는지는 다른 문제입니다. 먼저 프로그램이 붙인 세 가지 유형을 자료 생성 시 정해 둔 기준 유형과 행별로 비교합니다.</p><h4>시작 전 준비</h4><p>각 미션의 STEP 1~5는 앞 단계 결과를 이어서 사용합니다. WEL01_m03_student.ipynb의 첫 실행(환경 준비) 셀과 이 미션의 준비 셀을 순서대로 실행하면 현재 참고 프로그램의 결과가 raw라는 입력 표로 만들어집니다. PARAMETERS는 기본값을 유지합니다. 이 단계만 다시 실행하려면 그 미션의 준비 셀부터 바로 앞 STEP까지 순서대로 실행해 둡니다.</p><p>이번 입력 표는 <code>step1</code>, 완성한 결과 표는 <code>step2</code>입니다. 새로 만들 열은 <code>유형_일치</code>이며 배정 시간은 30분입니다. 노트북의 <code>M03 STEP 02</code> 코드 셀에서 TODO 호출을 본인이 작성한 pandas 계산으로 바꾸세요. 결과는 DataFrame이어야 합니다.</p><h4>입력값의 뜻</h4><ul><li><code>기초_경보유형</code> — STEP 1에서 계산한 heat·cold·normal 중 하나입니다.</li><li><code>rtype</code> — 합성 자료를 만들 때 지정한 기준 유형입니다. 실제 환자의 진단 결과가 아닙니다.</li></ul><h4>수행 순서</h4><ol><li>step1의 각 행에서 기초_경보유형과 rtype을 나란히 확인합니다.</li><li>두 문자열이 정확히 같으면 True, 다르면 False로 표시합니다.</li><li>유형_일치 열을 추가한 표를 step2에 저장합니다. 원래 유형 문자열을 고쳐 일치시키지 않습니다.</li><li>True 개수/전체 행 수로 유형 정확도를 확인하고, 불일치 행에서 예측 유형과 기준 유형을 각각 기록합니다.</li></ol><h4>결과를 읽는 방법</h4><p>True는 세 가지 유형 중 해당 행의 유형을 맞혔다는 뜻입니다. False에는 경보 누락뿐 아니라 열·한랭을 서로 바꾼 경우도 포함됩니다.</p><h4>직접 확인할 가상 예시</h4><p>다음은 규칙을 이해하기 위한 가상 사례입니다. 실제 실행값을 대신 제출하는 자료가 아닙니다.</p><table><thead><tr><th>가정한 입력</th><th>기대 결과</th><th>이유</th></tr></thead><tbody><tr><td><code>{"기초_경보유형": "heat", "rtype": "heat"}</code></td><td><code>True</code></td><td>유형이 같습니다.</td></tr><tr><td><code>{"기초_경보유형": "heat", "rtype": "cold"}</code></td><td><code>False</code></td><td>경보는 있지만 유형은 다릅니다.</td></tr></tbody></table><h4>완료 전 확인과 기록</h4><p>원행ID는 계산 전후 같은 자료를 찾는 번호입니다. 입력의 행 수·행 순서·인덱스·원래 열의 값을 유지하고, 앞 단계에서 만든 열도 모두 남깁니다. 각 행의 계산 결과는 새 열에 넣습니다. 실제 불리언 True·False와 문자열 "True"·"False"는 다릅니다. 값이 없다는 뜻의 결측값을 False나 0으로 대신 채우지 않습니다.</p><p>계산 뒤 해당 출력 열과 입력 열을 원행ID로 함께 확인합니다. 실제 자료에서 해당 사례가 있으면 최소 한 행의 입력·출력·판단 이유를 기록하고, 해당 사례가 없으면 0건 집계와 아래 가상 예시 시험 결과를 남깁니다. 관찰 내용은 5단계 완료 후 HTML의 해당 STEP 기록란에 옮기고 기록 JSON 저장을 눌러 보관합니다. HTML 입력란의 글은 페이지를 닫기 전에 저장해야 합니다.</p><h4>주의할 점</h4><p>normal과 heat를 모두 위험 여부라는 이유로 묶어서 비교하지 않습니다. 위험을 찾아냈는지는 다음 단계의 별도 계산입니다.</p>


> **교수자·해설용 FAIL 점검 포인트**: 위험 경보가 있다는 이유만으로 유형_일치를 True로 만들거나 전체 정확도를 모든 행에 반복해서 넣으면 FAIL입니다.

In [ ]:
# M03 STEP 02 — 계산한 경보 유형이 합성 자료의 기준 유형과 같은지 확인하기 (교수자·해설용 예시 풀이)
# 해설: 기초_경보유형과 rtype 문자열이 정확히 같을 때만 True입니다.
step2 = step1.copy()
step2["유형_일치"] = (step2["기초_경보유형"] == step2["rtype"]).astype(bool)
display(step2[["원행ID", "유형_일치"]].head(10))

### M03 STEP 03 — 위험 구간을 실제로 찾아낸 행 표시하기

<h4>이 단계에서 해결할 일</h4><p>유형을 맞히지 못했더라도 위험 구간에서 경보가 나온 경우는 위험 탐지에 포함됩니다. 이번 단계에서는 열·한랭 구분을 잠시 내려놓고 위험 있음·없음의 두 범주로 참양성(TP)을 계산합니다.</p><h4>시작 전 준비</h4><p>각 미션의 STEP 1~5는 앞 단계 결과를 이어서 사용합니다. WEL01_m03_student.ipynb의 첫 실행(환경 준비) 셀과 이 미션의 준비 셀을 순서대로 실행하면 현재 참고 프로그램의 결과가 raw라는 입력 표로 만들어집니다. PARAMETERS는 기본값을 유지합니다. 이 단계만 다시 실행하려면 그 미션의 준비 셀부터 바로 앞 STEP까지 순서대로 실행해 둡니다.</p><p>이번 입력 표는 <code>step2</code>, 완성한 결과 표는 <code>step3</code>입니다. 새로 만들 열은 <code>위험_TP</code>이며 배정 시간은 30분입니다. 노트북의 <code>M03 STEP 03</code> 코드 셀에서 TODO 호출을 본인이 작성한 pandas 계산으로 바꾸세요. 결과는 DataFrame이어야 합니다.</p><h4>입력값의 뜻</h4><ul><li><code>기초_경보유형</code> — normal 이외의 heat·cold는 위험 경보가 나온 경우입니다.</li><li><code>stage</code> — 합성 자료의 기준 단계입니다. 0은 정상, 1은 주의, 2는 위험이며 이 실습에서는 1 이상을 탐지 대상으로 묶습니다.</li></ul><h4>수행 순서</h4><ol><li>step2에서 기초_경보유형이 normal이 아닌지 확인합니다.</li><li>같은 행의 stage가 1 이상인지 확인합니다.</li><li>두 조건을 모두 만족할 때만 위험_TP=True로 하고 step3에 저장합니다.</li><li>True 행 수를 TP로 집계합니다. 유형_일치=False이면서 위험_TP=True인 행이 있으면 두 판단이 다른 이유를 적습니다.</li></ol><h4>결과를 읽는 방법</h4><p>TP는 합성 기준에서 주의·위험인 구간에 어떤 위험 경보든 나온 경우입니다. 열·한랭 유형까지 맞혔다는 뜻은 아닙니다.</p><h4>직접 확인할 가상 예시</h4><p>다음은 규칙을 이해하기 위한 가상 사례입니다. 실제 실행값을 대신 제출하는 자료가 아닙니다.</p><table><thead><tr><th>가정한 입력</th><th>기대 결과</th><th>이유</th></tr></thead><tbody><tr><td><code>{"기초_경보유형": "heat", "stage": 1}</code></td><td><code>True</code></td><td>주의도 탐지 대상입니다.</td></tr><tr><td><code>{"기초_경보유형": "normal", "stage": 2}</code></td><td><code>False</code></td><td>위험 구간이지만 경보가 없습니다.</td></tr><tr><td><code>{"기초_경보유형": "cold", "stage": 0}</code></td><td><code>False</code></td><td>기준이 정상이므로 TP가 아닙니다.</td></tr></tbody></table><h4>완료 전 확인과 기록</h4><p>원행ID는 계산 전후 같은 자료를 찾는 번호입니다. 입력의 행 수·행 순서·인덱스·원래 열의 값을 유지하고, 앞 단계에서 만든 열도 모두 남깁니다. 각 행의 계산 결과는 새 열에 넣습니다. 실제 불리언 True·False와 문자열 "True"·"False"는 다릅니다. 값이 없다는 뜻의 결측값을 False나 0으로 대신 채우지 않습니다.</p><p>계산 뒤 해당 출력 열과 입력 열을 원행ID로 함께 확인합니다. 실제 자료에서 해당 사례가 있으면 최소 한 행의 입력·출력·판단 이유를 기록하고, 해당 사례가 없으면 0건 집계와 아래 가상 예시 시험 결과를 남깁니다. 관찰 내용은 5단계 완료 후 HTML의 해당 STEP 기록란에 옮기고 기록 JSON 저장을 눌러 보관합니다. HTML 입력란의 글은 페이지를 닫기 전에 저장해야 합니다.</p><h4>주의할 점</h4><p>stage를 실제 의무 판정으로 설명하지 않습니다. stage=1도 포함하며 stage=2만 세지 않습니다.</p>


> **교수자·해설용 FAIL 점검 포인트**: rtype 일치 여부를 TP 조건에 추가하거나 stage=1을 빼거나 경보 없는 행을 TP로 세면 FAIL입니다.

In [ ]:
# M03 STEP 03 — 위험 구간을 실제로 찾아낸 행 표시하기 (교수자·해설용 예시 풀이)
# 해설: 경보 발생(기초_경보유형 != 'normal')이면서 기준 stage >= 1(주의 포함)인 행이 위험_TP=True입니다.
step3 = step2.copy()
step3["위험_TP"] = (
    step3["기초_경보유형"].notna()
    & (step3["기초_경보유형"] != "normal")
    & (step3["stage"] >= 1)
).astype(bool)
display(step3[["원행ID", "위험_TP"]].head(10))

### M03 STEP 04 — 정상 구간에 발생한 오경보 표시하기

<h4>이 단계에서 해결할 일</h4><p>정상 구간에 불필요한 경보가 반복되면 확인 업무가 늘어납니다. 어떤 위험 경보가 나왔지만 합성 기준 단계는 정상인 행을 골라 오경보(FP)를 계산합니다.</p><h4>시작 전 준비</h4><p>각 미션의 STEP 1~5는 앞 단계 결과를 이어서 사용합니다. WEL01_m03_student.ipynb의 첫 실행(환경 준비) 셀과 이 미션의 준비 셀을 순서대로 실행하면 현재 참고 프로그램의 결과가 raw라는 입력 표로 만들어집니다. PARAMETERS는 기본값을 유지합니다. 이 단계만 다시 실행하려면 그 미션의 준비 셀부터 바로 앞 STEP까지 순서대로 실행해 둡니다.</p><p>이번 입력 표는 <code>step3</code>, 완성한 결과 표는 <code>step4</code>입니다. 새로 만들 열은 <code>오경보_FP</code>이며 배정 시간은 30분입니다. 노트북의 <code>M03 STEP 04</code> 코드 셀에서 TODO 호출을 본인이 작성한 pandas 계산으로 바꾸세요. 결과는 DataFrame이어야 합니다.</p><h4>입력값의 뜻</h4><ul><li><code>기초_경보유형</code> — heat 또는 cold이면 경보가 발생한 것입니다.</li><li><code>stage</code> — 0인 행이 합성 기준의 정상 구간입니다.</li></ul><h4>수행 순서</h4><ol><li>step3에서 경보 유형이 normal이 아닌 행을 찾습니다.</li><li>같은 행의 stage가 0인지 확인합니다.</li><li>두 조건을 모두 만족하면 오경보_FP=True로 만들고 step4에 저장합니다.</li><li>True 개수를 집계하고 해당 행의 체온·심박·경보 유형·stage를 함께 확인합니다. 어떤 규칙이 경보를 만들었는지 기록합니다.</li></ol><h4>결과를 읽는 방법</h4><p>FP는 정상 기준 구간에 경보가 발생한 경우입니다. 경보 전체에서 차지하는 비율과 정상 구간 중 발생 비율은 서로 다릅니다.</p><h4>직접 확인할 가상 예시</h4><p>다음은 규칙을 이해하기 위한 가상 사례입니다. 실제 실행값을 대신 제출하는 자료가 아닙니다.</p><table><thead><tr><th>가정한 입력</th><th>기대 결과</th><th>이유</th></tr></thead><tbody><tr><td><code>{"기초_경보유형": "heat", "stage": 0}</code></td><td><code>True</code></td><td>정상 구간에 경보가 발생했습니다.</td></tr><tr><td><code>{"기초_경보유형": "heat", "stage": 1}</code></td><td><code>False</code></td><td>합성 기준에서도 탐지 대상입니다.</td></tr><tr><td><code>{"기초_경보유형": "normal", "stage": 0}</code></td><td><code>False</code></td><td>경보가 없으므로 FP가 아닙니다.</td></tr></tbody></table><h4>완료 전 확인과 기록</h4><p>원행ID는 계산 전후 같은 자료를 찾는 번호입니다. 입력의 행 수·행 순서·인덱스·원래 열의 값을 유지하고, 앞 단계에서 만든 열도 모두 남깁니다. 각 행의 계산 결과는 새 열에 넣습니다. 실제 불리언 True·False와 문자열 "True"·"False"는 다릅니다. 값이 없다는 뜻의 결측값을 False나 0으로 대신 채우지 않습니다.</p><p>계산 뒤 해당 출력 열과 입력 열을 원행ID로 함께 확인합니다. 실제 자료에서 해당 사례가 있으면 최소 한 행의 입력·출력·판단 이유를 기록하고, 해당 사례가 없으면 0건 집계와 아래 가상 예시 시험 결과를 남깁니다. 관찰 내용은 5단계 완료 후 HTML의 해당 STEP 기록란에 옮기고 기록 JSON 저장을 눌러 보관합니다. HTML 입력란의 글은 페이지를 닫기 전에 저장해야 합니다.</p><h4>주의할 점</h4><p>FAR를 추가로 계산할 때에는 정상 구간 수(FP+TN)를 분모로 씁니다. 전체 경보 수를 분모로 사용하지 않습니다.</p>


> **교수자·해설용 FAIL 점검 포인트**: 경보 유형이 틀렸다는 이유만으로 모든 유형 불일치를 FP로 세거나 정상 분모와 경보 분모를 혼동하면 FAIL입니다.

In [ ]:
# M03 STEP 04 — 정상 구간에 발생한 오경보 표시하기 (교수자·해설용 예시 풀이)
# 해설: 경보 발생(기초_경보유형 != 'normal')이면서 기준 stage == 0(정상 구간)인 행이 오경보_FP=True입니다.
step4 = step3.copy()
step4["오경보_FP"] = (
    step4["기초_경보유형"].notna()
    & (step4["기초_경보유형"] != "normal")
    & (step4["stage"] == 0)
).astype(bool)
display(step4[["원행ID", "오경보_FP"]].head(10))

### M03 STEP 05 — 경보가 나오지 않은 위험 구간 찾아 설명하기

<h4>이 단계에서 해결할 일</h4><p>전체 정확도가 높아도 실제로 찾아야 할 구간을 놓칠 수 있습니다. 합성 기준은 주의·위험인데 프로그램에는 경보가 없는 행을 찾아 미탐(FN)으로 표시하고, 놓친 이유를 원래 입력값에서 확인합니다.</p><h4>시작 전 준비</h4><p>각 미션의 STEP 1~5는 앞 단계 결과를 이어서 사용합니다. WEL01_m03_student.ipynb의 첫 실행(환경 준비) 셀과 이 미션의 준비 셀을 순서대로 실행하면 현재 참고 프로그램의 결과가 raw라는 입력 표로 만들어집니다. PARAMETERS는 기본값을 유지합니다. 이 단계만 다시 실행하려면 그 미션의 준비 셀부터 바로 앞 STEP까지 순서대로 실행해 둡니다.</p><p>이번 입력 표는 <code>step4</code>, 완성한 결과 표는 <code>step5</code>입니다. 새로 만들 열은 <code>미탐_FN</code>이며 배정 시간은 30분입니다. 노트북의 <code>M03 STEP 05</code> 코드 셀에서 TODO 호출을 본인이 작성한 pandas 계산으로 바꾸세요. 결과는 DataFrame이어야 합니다.</p><h4>입력값의 뜻</h4><ul><li><code>기초_경보유형</code> — normal이면 이 규칙의 경보가 나오지 않았습니다.</li><li><code>stage</code> — 1 이상이면 탐지 대상으로 정한 구간입니다.</li><li><code>rtype·temp5·hr5</code> — 미탐의 유형과 발생 조건을 설명할 때 확인할 원자료입니다.</li></ul><h4>수행 순서</h4><ol><li>step4에서 기초_경보유형이 normal이면서 stage가 1 이상인 행을 찾습니다.</li><li>해당 행에만 미탐_FN=True를 넣어 step5에 저장합니다.</li><li>미탐을 rtype별로 세고, 열 유형에서는 심박 조건 때문에 제외된 사례가 있는지 원래 수치로 확인합니다.</li><li>미탐 행의 원행ID·체온·심박·기준 유형·놓친 조건을 HTML 기록란에 적습니다. 임계값을 바꿔 보려면 기본 실행과 다른 실험으로 남깁니다.</li></ol><h4>결과를 읽는 방법</h4><p>미탐_FN=True는 계산 실패 표시가 아니라 규칙이 해당 위험 구간을 놓쳤다는 결과입니다. 정확히 찾아낸 미탐을 숨기지 않습니다.</p><h4>직접 확인할 가상 예시</h4><p>다음은 규칙을 이해하기 위한 가상 사례입니다. 실제 실행값을 대신 제출하는 자료가 아닙니다.</p><table><thead><tr><th>가정한 입력</th><th>기대 결과</th><th>이유</th></tr></thead><tbody><tr><td><code>{"기초_경보유형": "normal", "stage": 1}</code></td><td><code>True</code></td><td>주의 구간인데 경보가 없습니다.</td></tr><tr><td><code>{"기초_경보유형": "cold", "stage": 2}</code></td><td><code>False</code></td><td>위험 경보가 이미 있습니다.</td></tr><tr><td><code>{"기초_경보유형": "normal", "stage": 0}</code></td><td><code>False</code></td><td>정상 기준 구간입니다.</td></tr></tbody></table><h4>완료 전 확인과 기록</h4><p>원행ID는 계산 전후 같은 자료를 찾는 번호입니다. 입력의 행 수·행 순서·인덱스·원래 열의 값을 유지하고, 앞 단계에서 만든 열도 모두 남깁니다. 각 행의 계산 결과는 새 열에 넣습니다. 실제 불리언 True·False와 문자열 "True"·"False"는 다릅니다. 값이 없다는 뜻의 결측값을 False나 0으로 대신 채우지 않습니다.</p><p>계산 뒤 해당 출력 열과 입력 열을 원행ID로 함께 확인합니다. 실제 자료에서 해당 사례가 있으면 최소 한 행의 입력·출력·판단 이유를 기록하고, 해당 사례가 없으면 0건 집계와 아래 가상 예시 시험 결과를 남깁니다. 관찰 내용은 5단계 완료 후 HTML의 해당 STEP 기록란에 옮기고 기록 JSON 저장을 눌러 보관합니다. HTML 입력란의 글은 페이지를 닫기 전에 저장해야 합니다.</p><h4>주의할 점</h4><p>놓친 위험 구간을 정상으로 다시 이름 붙이지 않습니다. 실제 도움 요청 여부는 교육용 점수만으로 결정하지 않습니다.</p>


> **교수자·해설용 FAIL 점검 포인트**: 위험 구간 전체를 미탐으로 세거나, stage=1을 제외하거나, 미탐을 없애려고 기준 자료를 수정하면 FAIL입니다.

In [ ]:
# M03 STEP 05 — 경보가 나오지 않은 위험 구간 찾아 설명하기 (교수자·해설용 예시 풀이)
# 해설: 경보 미발생(기초_경보유형 == 'normal')이면서 기준 stage >= 1인 행이 미탐_FN=True입니다.
step5 = step4.copy()
step5["미탐_FN"] = ((step5["기초_경보유형"] == "normal") & (step5["stage"] >= 1)).astype(bool)
display(step5[["원행ID", "미탐_FN"]].head(10))

### 마무리: 결과 화면 생성과 기록 저장

아래 셀을 실행하면 다섯 단계의 실제 계산 결과가 HTML로 저장됩니다. 왼쪽 STEP 메뉴로 결과를 확인하고 관찰 기록을 입력하세요. 기록 JSON을 내려받아야 입력한 글을 별도로 보관할 수 있습니다. 구조 검사 완료는 교관의 최종 합격 판정이 아닙니다.


In [ ]:
STAGE_SPECS = SOURCE_BUNDLE["missions"][MID]["stages"]
result_html = finish_mission(MID, raw, [step1, step2, step3, step4, step5], STAGE_SPECS)
